In [3]:
import math
import subprocess
import numpy as np
import meep as mp
import re
from meep import mpb
from IPython.utils.capture import capture_output
from tqdm import tqdm
import time
import matplotlib.pyplot as plt
import pandas as pd
from pandas import Series, DataFrame

/Users/henry/opt/anaconda3/envs/mp/lib/python3.11/site-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/Users/henry/opt/anaconda3/envs/mp/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [7]:
# for cylinder
def atlas_map(radius,epsilon):

    num_bands = 8
    # k_points = Bloch wavevectors
    k_points = [mp.Vector3(),          # Gamma
                mp.Vector3(0.5),       # X
                mp.Vector3(0.5, 0.5),  # M
                mp.Vector3()]          # Gamma
    k_points = mp.interpolate(4, k_points)
    geometry = [mp.Cylinder(radius, material=mp.Medium(epsilon=epsilon))]
    geometry_lattice = mp.Lattice(size=mp.Vector3(1, 1))
    resolution = 32
    ms = mpb.ModeSolver(num_bands=num_bands,
                    k_points=k_points,
                    geometry=geometry,
                    geometry_lattice=geometry_lattice,
                    resolution=resolution)
    with capture_output() as output:
        ms.run_tm()
    variable_containing_output = output.stdout.strip()
    pattern = r'band\s+\d+\s*\(\s*([\d.]+)\s*\)'
    matches = re.findall(pattern, variable_containing_output)
    matches=[eval(i) for i in matches]
    return matches

In [29]:
def extract(lst,n):
    return list(list(zip(*lst))[n])
def data_atlas(epsilon):
    radius = np.arange(0,0.5,0.05)
    freq=[]
    for r in tqdm(radius):
        freq.append(atlas_map(r,epsilon))  
    for i in range(0,len(freq)):
        if(freq[i]==[]):
            freq[i]=[0,0]
    col3 = []
    col4 = []
    col5 = []
    for i in range(len(freq)):
        col3.append(freq[i][0])
        col4.append(freq[i][1])
        col5.append(freq[i][1]-freq[i][0])
    col1 = [epsilon]*len(radius)
    col2 = radius
    return [col1,col2,col3,col4]

In [30]:
col1 = []
col2 = []
col3 = []
col4 = []
col5 = []
for eps in range(3,20):
    data = data_atlas(eps)
    col1.extend(data[0])
    col2.extend(data[1])
    col3.extend(data[2])
    col4.extend(data[3])
    col5.extend(data[4])

100%|███████████████████████████████████████████| 50/50 [00:17<00:00,  2.87it/s]


In [31]:
    saved_data = DataFrame({
        "epsilon": col1,
        "radius": col2,
        "max(w)": col3,
        "min(w)": col4,
        "differences": col5
    })

In [32]:
print(saved_data)

     epsilon  radius  max(w)  min(w)
0          3    0.00     0.0     0.0
1          3    0.01     0.0     0.0
2          3    0.02     0.0     0.0
3          3    0.03     0.0     0.0
4          3    0.04     0.0     0.0
..       ...     ...     ...     ...
845       19    0.45     0.0     0.0
846       19    0.46     0.0     0.0
847       19    0.47     0.0     0.0
848       19    0.48     0.0     0.0
849       19    0.49     0.0     0.0

[850 rows x 4 columns]


In [33]:
saved_data.to_csv('diamond.csv', index=False)